# Quickstart

The 90% path: pick a time lattice, build a circuit, push inputs, read outputs through an `Evaluator`.

## Running sum — build your own

`Input → Integrate → output` on a 1-D lattice. `Evaluator.at(t)` materialises the stream at a timestamp.

In [ ]:
from pydbsp.core import Time1D
from pydbsp.stream import Input
from pydbsp.stream.operators.linear import Integrate
from pydbsp.evaluator import Evaluator
from pydbsp.zset import ZSet, ZSetAddition

g = ZSetAddition[int]()
inp = Input(g, Time1D)
cum = Integrate(inp, Time1D, axis=0)
ev = Evaluator(cum)

inp.push(Time1D.at(0), ZSet({1: 1}))
inp.push(Time1D.at(1), ZSet({2: 1}))
inp.push(Time1D.at(2), ZSet({1: 1}))

[ev.at(Time1D.at(k)).inner for k in range(3)]

## Graph reachability — reach for a canned algorithm

`pydbsp.algorithms.*` bundles the common DBSP circuits. Edges are a 1-D input (they vary across outer ticks, constant within the inner fixpoint); the circuit introduces them into the 2-D body lattice internally. Push initial edges, call `saturate_reach` to drive the inner fixpoint, read the closure at the outer timestamp.

In [ ]:
from pydbsp.algorithms.reachability_indexed import (
    IncrementalReachabilityWithIndexing,
    saturate_reach,
)
from pydbsp.core import Time1D
from pydbsp.zset import ZSet

c = IncrementalReachabilityWithIndexing()
c.edges.push(Time1D.at(0), ZSet({(1, 2): 1, (2, 3): 1, (3, 4): 1}))
saturate_reach(c, outer_tick=0)

c.observable_at((0,)).inner